## Final Inference German Data Creation
This notebook executes the final preparation stage for the German evaluation dataset. It transforms a random sample of German news articles into a high-quality, sentence-level corpus suitable for fine-grained translation evaluation.

### Key Achievements:
* **Statistical Sampling:**
  * Selected a random subset of **500 records** from the massive 167k+ dataset to create a manageable but representative evaluation set.
* **Linguistic Segmentation (Spacy):**
  * Utilized the `spacy` library (`de_core_news_sm`) to linguistically split complex German paragraphs into individual sentences.
  * This is critical because NLLB (and most NMT models) performs significantly better on sentence-level inputs than on long, unstructured document chunks.
* **Granular Data Expansion:**
  * Converted the 500 source records into **7,604 clean, standalone sentences**.
  * Applied rigorous filtering (min/max length constraints) to remove noise like headlines, captions, or menu text, ensuring only translatable content remains.
* **Inference Readiness:**
  * Automatically re-injected the specific prompt `"translate German to Odia: "` to every single sentence, making the final file `german_news_clean_final.jsonl` ready for immediate feeding into the translation model.
  
### Workflow Context:
* **Input:** `german_news_inference.jsonl` (167k records)
* **Process:** Random Sample $\rightarrow$ Spacy Sentence Split $\rightarrow$ Filter
* **Output:** `german_news_clean_final.jsonl` (7,604 Sentences)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json
import random
import re
import spacy
from tqdm import tqdm

In [ ]:
!pip install -U spacy

In [ ]:
!python -m spacy download de_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 113.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# Input and Output paths
input_file = "/content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/german_news_inference.jsonl"
output_file = "/content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/german_news_eval_subset.jsonl"

# Configuration
SAMPLE_SIZE = 500       # 500 random records is enough for evaluation
MAX_CHAR_LENGTH = 2500  # Approx 600-700 tokens, leaving room for translation

In [ ]:
print(f"Reading {input_file}...")
all_records = []

# Read the file
with open(input_file, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        if data.get('input_text'):
            all_records.append(data)

print(f"Total records found: {len(all_records)}")

Reading /content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/german_news_inference.jsonl...
Total records found: 167280


In [ ]:
# 1. Randomly Sample
random.seed(42) # specific seed for reproducibility
if len(all_records) > SAMPLE_SIZE:
    sampled_records = random.sample(all_records, SAMPLE_SIZE)
else:
    sampled_records = all_records

print(f"Selected {len(sampled_records)} records for evaluation.")

Selected 500 records for evaluation.


In [ ]:
# 2. Truncate Length
cleaned_records = []
for record in sampled_records:
    text = record['input_text']

    # Truncate if too long (keep first 2500 chars)
    if len(text) > MAX_CHAR_LENGTH:
        text = text[:MAX_CHAR_LENGTH] + "..."

    record['input_text'] = text
    cleaned_records.append(record)

In [ ]:
# Save to new file
with open(output_file, 'w', encoding='utf-8') as f:
    for record in cleaned_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Success! Saved valid subset to '{output_file}'. Use this file for inference.")

Success! Saved valid subset to '/content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/german_news_eval_subset.jsonl'. Use this file for inference.


In [ ]:
print(cleaned_records[:3])

[{'input_text': 'translate German to Odia: Lotto-Rekordgewinn - Tipperin aus Baden-Württemberg räumt 42,5 Millionen Euro ab: Mit rund 42,5 Millionen Euro hat eine Frau aus Baden-Württemberg den höchsten Lotto-Gewinn bei einer Ziehung 6aus49 in Deutschland geholt. Das teilte Lotto am Montag in Stuttgart mit.\n\nDie Tipperin komme aus dem Zollernalbkreis und habe bei der Samstagsziehung alle sechs Gewinnzahlen richtig getippt sowie die passende Superzahl gehabt. Ihre Glückzahlen waren die 4, 8, 16, 22, 28 und 33, die Superzahl war die 6. Gewinner habe es laut einer Meldung von "RTL" jedoch auch in anderen Bundesländern gegeben. So seien knapp 700.000 Euro an Tipper aus Baden-Württemberg, Bayern, Nordrhein-Westfalen und Niedersachen gegangen.\n\nChance auf Jackpot bei 1 zu 140 Millionen\n\nLotto zählt in Deutschland zu den beliebtesten Glücksspielen. Beim Spiel "6 aus 49" müssen die Teilnehmer sechs Zahlen aus einer von 1 bis 49 bestehenden Zahlenreihe auf einem Tippschein ankreuzen. Je m

In [ ]:
output_file_new = "/content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/german_news_clean_final.jsonl"

In [ ]:
PREFIX = "translate German to Odia: "

In [ ]:
# Load German NLP model
print("Loading German language model...")
nlp = spacy.load("de_core_news_sm")

Loading German language model...


In [ ]:
print(f"Processing {output_file}...")

cleaned_sentences = []
processed_articles = 0

with open(output_file, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        raw_text = data.get('input_text', '')

        # 1. Remove Prefix to process raw German
        if raw_text.startswith(PREFIX):
            raw_text = raw_text[len(PREFIX):]

        # 2. Pre-cleaning (Removing obvious artifacts before splitting)
        # Remove "Anzeige" if it appears alone or with spaces
        raw_text = re.sub(r'\bAnzeige\b', '', raw_text)

        # 3. Use spaCy to split into sentences
        doc = nlp(raw_text)

        # Track seen sentences per article to remove duplicate headers
        seen_in_article = set()

        for sent in doc.sents:
            text = sent.text.strip()

            # --- FILTERS ---

            # A. Length Filter: Skip empty or very short junk (e.g. "Und:", "S. 4")
            if len(text) < 20:
                continue

            # B. Duplicate Filter: If header appears twice, skip subsequent ones
            if text in seen_in_article:
                continue

            # C. Artifact Filter: Skip lines that look like bylines or metadata
            if text.startswith(("Kommentar von", "Von", "Bild:", "Foto:")):
                continue

            # D. Completeness Filter: Must end with punctuation
            # This kills the "Schul..." error
            if not text.endswith(('.', '?', '!', '"', '”', '’')):
                continue

            # --- ADD TO LIST ---
            seen_in_article.add(text)

            # Create new JSON object for this sentence
            new_entry = {
                "input_text": PREFIX + text,
                "target_text": "",
                "original_source": data.get('original_source', 'N/A'),
                "article_context": data.get('title', '') # Optional: keep title for reference
            }
            cleaned_sentences.append(new_entry)

        processed_articles += 1

Processing /content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/german_news_eval_subset.jsonl...


In [ ]:
# Save the new sentence-level dataset
with open(output_file_new, 'w', encoding='utf-8') as f:
    for entry in cleaned_sentences:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

In [ ]:
print(f"\nProcessing Complete!")
print(f"Parsed {processed_articles} articles.")
print(f"Generated {len(cleaned_sentences)} clean sentences.")
print(f"Saved to: {output_file_new}")


Processing Complete!
Parsed 500 articles.
Generated 7604 clean sentences.
Saved to: /content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/german_news_clean_final.jsonl


In [ ]:
# Preview
print("\n--- Sample Output ---")
for i in range(3):
    print(json.dumps(cleaned_sentences[i], indent=2, ensure_ascii=False))


--- Sample Output ---
{
  "input_text": "translate German to Odia: Mit rund 42,5 Millionen Euro hat eine Frau aus Baden-Württemberg den höchsten Lotto-Gewinn bei einer Ziehung 6aus49 in Deutschland geholt.",
  "target_text": "",
  "original_source": "focus",
  "article_context": ""
}
{
  "input_text": "translate German to Odia: Das teilte Lotto am Montag in Stuttgart mit.",
  "target_text": "",
  "original_source": "focus",
  "article_context": ""
}
{
  "input_text": "translate German to Odia: Die Tipperin komme aus dem Zollernalbkreis und habe bei der Samstagsziehung alle sechs Gewinnzahlen richtig getippt sowie die passende Superzahl gehabt.",
  "target_text": "",
  "original_source": "focus",
  "article_context": ""
}
